<a href="https://colab.research.google.com/github/edwintorrecilla-create/control-cedulas/blob/main/entregas_almacen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.chart import PieChart, Reference

def generar_reporte_almacen():
    # =========================================================================
    # PASO 1: Configuración de Directorios y Carpetas
    # =========================================================================
    # Definimos los nombres de las carpetas de entrada y salida
    folder_entrada = "entrada"
    folder_salida = "salida"

    # os.makedirs crea la carpeta si no existe previamente
    os.makedirs(folder_entrada, exist_ok=True)
    os.makedirs(folder_salida, exist_ok=True)

    # Rutas relativas de los archivos de Excel
    path_entrada = os.path.join(folder_entrada, "Entregas Almacen Junio.xlsx")
    path_salida = os.path.join(folder_salida, "Reporte_Entregas_Categorizado.xlsx")

    # Verificación de seguridad
    if not os.path.exists(path_entrada):
        print(f"⚠️ Atención: Coloca tu archivo de Excel dentro de la carpeta '{folder_entrada}' con el nombre 'Entregas Almacen Junio.xlsx'")
        return

    print("📖 Paso 1: Leyendo y cargando los datos iniciales...")
    # pandas.read_excel lee la hoja de cálculo y la convierte en un DataFrame (matriz de datos)
    df = pd.read_excel(path_entrada)

    # =========================================================================
    # PASO 2: Categorización de Artículos por Prefijo
    # =========================================================================
    # Diccionario de equivalencias: asigna el nombre formal según los primeros 2 caracteres
    mapa_categorias = {
        'RT': 'Repuestos de Maquina',
        'DT': 'Dotación',
        'CN': 'Consumibles',
        'ES': 'Elementos de Izaje',
        'SG': 'EPPS',
        'HT': 'Herramientas'
    }

    # Creamos una columna 'Prefijo' extrayendo las primeras 2 letras de 'Número de artículo'
    df['Prefijo'] = df['Número de artículo'].astype(str).str.strip().str[:2].str.upper()

    # Asignamos la categoría correspondiente usando el diccionario
    df['Categoría'] = df['Prefijo'].map(mapa_categorias).fillna('Otros / Sin Categoría')

    # =========================================================================
    # PASO 3: Agrupamiento Financiero y Conteo de Entregas
    # =========================================================================
    print("📊 Paso 2: Calculando sumatorias y porcentajes en dinero...")
    # Groupby agrupa por categoría y aplica funciones de agregación
    resumen = df.groupby('Categoría').agg(
        Suma_Dinero=('Total', 'sum'),                    # Suma monetaria en pesos
        Entregas_Unicas=('Número de documento', 'nunique'), # Vales de almacén únicos
        Cantidad_Registros=('Número de documento', 'count') # Cantidad de filas
    ).reset_index()

    # Ordenamos de mayor a menor monto económico
    resumen = resumen.sort_values(by='Suma_Dinero', ascending=False)

    # =========================================================================
    # PASO 4: Construcción del Libro de Excel con OpenPyXL
    # =========================================================================
    print("🎨 Paso 3: Diseñando el reporte en Excel con gráfico y tablas...")
    wb = Workbook()

    # --- Hoja 1: Resumen y Gráfico ---
    ws_resumen = wb.active
    ws_resumen.title = "Resumen por Categoría"
    ws_resumen.views.sheetView[0].showGridLines = True # Mostrar cuadrícula

    # Definición de colores y estilos corporativos
    color_encabezado = PatternFill(start_color="1F497D", end_color="1F497D", fill_type="solid") # Azul oscuro
    fuente_encabezado = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
    color_alternado = PatternFill(start_color="F2F5F9", end_color="F2F5F9", fill_type="solid")  # Azul muy claro
    color_totales = PatternFill(start_color="DCE6F1", end_color="DCE6F1", fill_type="solid")    # Gris-azul

    borde_fino = Border(
        left=Side(style='thin', color='D9D9D9'),
        right=Side(style='thin', color='D9D9D9'),
        top=Side(style='thin', color='D9D9D9'),
        bottom=Side(style='thin', color='D9D9D9')
    )

    # Banner del Título Principal
    ws_resumen.merge_cells("A1:E1")
    celda_titulo = ws_resumen["A1"]
    celda_titulo.value = "REPORTE DE ENTREGAS POR CATEGORÍA - PORCENTAJE EN DINERO"
    celda_titulo.font = Font(name="Calibri", size=13, bold=True, color="FFFFFF")
    celda_titulo.fill = color_encabezado
    celda_titulo.alignment = Alignment(horizontal="center", vertical="center")
    ws_resumen.row_dimensions[1].height = 35

    ws_resumen["A2"] = "Consolidado Almacén General - Análisis del Periodo"
    ws_resumen["A2"].font = Font(name="Calibri", size=10, italic=True, color="595959")

    # Encabezados de la tabla
    encabezados = [
        "Categoría",
        "Total Dinero ($)",
        "Participación en Dinero (%)",
        "Número de Entregas (Docs Únicos)",
        "Líneas Registradas"
    ]

    fila_inicio = 4
    for idx_col, texto in enumerate(encabezados, 1):
        celda = ws_resumen.cell(row=fila_inicio, column=idx_col, value=texto)
        celda.fill = color_encabezado
        celda.font = fuente_encabezado
        celda.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws_resumen.row_dimensions[fila_inicio].height = 28

    # Calcular la fila donde quedará la suma total para usarla en las fórmulas
    fila_total_general = fila_inicio + len(resumen) + 1

    # Insertar los datos fila por fila
    fila_actual = fila_inicio + 1
    for _, fila in resumen.iterrows():
        c1 = ws_resumen.cell(row=fila_actual, column=1, value=fila['Categoría'])
        c2 = ws_resumen.cell(row=fila_actual, column=2, value=fila['Suma_Dinero'])

        # Fórmula Excel: Dividir el dinero de la categoría sobre el dinero total general
        formula_pct = f"=B{fila_actual}/B${fila_total_general}"
        c3 = ws_resumen.cell(row=fila_actual, column=3, value=formula_pct)

        c4 = ws_resumen.cell(row=fila_actual, column=4, value=fila['Entregas_Unicas'])
        c5 = ws_resumen.cell(row=fila_actual, column=5, value=fila['Cantidad_Registros'])

        # Formatos de número de Excel
        c2.number_format = '"$"#,##0.00'  # Moneda con decimales
        c3.number_format = '0.00%'       # Porcentaje con 2 decimales
        c4.number_format = '#,##0'       # Entero
        c5.number_format = '#,##0'       # Entero

        # Alineaciones visuales
        c1.alignment = Alignment(horizontal="left", vertical="center")
        c2.alignment = Alignment(horizontal="right", vertical="center")
        c3.alignment = Alignment(horizontal="right", vertical="center")
        c4.alignment = Alignment(horizontal="center", vertical="center")
        c5.alignment = Alignment(horizontal="center", vertical="center")

        # Filas con fondo alternado
        if (fila_actual - fila_inicio) % 2 == 0:
            for c in [c1, c2, c3, c4, c5]:
                c.fill = color_alternado

        for c in [c1, c2, c3, c4, c5]:
            c.border = borde_fino

        fila_actual += 1

    # Fila de Totales
    row_tot = fila_actual
    ws_resumen.cell(row=row_tot, column=1, value="TOTAL GENERAL").font = Font(name="Calibri", size=11, bold=True)

    celda_tot_sum = ws_resumen.cell(row=row_tot, column=2, value=f"=SUM(B{fila_inicio+1}:B{row_tot-1})")
    celda_tot_pct = ws_resumen.cell(row=row_tot, column=3, value=f"=SUM(C{fila_inicio+1}:C{row_tot-1})")
    celda_tot_ent = ws_resumen.cell(row=row_tot, column=4, value=df['Número de documento'].nunique())
    celda_tot_lin = ws_resumen.cell(row=row_tot, column=5, value=f"=SUM(E{fila_inicio+1}:E{row_tot-1})")

    for c in [celda_tot_sum, celda_tot_pct, celda_tot_ent, celda_tot_lin]:
        c.font = Font(name="Calibri", size=11, bold=True)

    celda_tot_sum.number_format = '"$"#,##0.00'
    celda_tot_pct.number_format = '0.00%'
    celda_tot_ent.number_format = '#,##0'
    celda_tot_lin.number_format = '#,##0'

    ws_resumen.cell(row=row_tot, column=1).alignment = Alignment(horizontal="left", vertical="center")
    celda_tot_sum.alignment = Alignment(horizontal="right", vertical="center")
    celda_tot_pct.alignment = Alignment(horizontal="right", vertical="center")
    celda_tot_ent.alignment = Alignment(horizontal="center", vertical="center")
    celda_tot_lin.alignment = Alignment(horizontal="center", vertical="center")

    borde_total = Border(
        left=Side(style='thin', color='D9D9D9'),
        right=Side(style='thin', color='D9D9D9'),
        top=Side(style='thin', color='000000'),
        bottom=Side(style='double', color='000000') # Doble línea inferior de cierre contable
    )
    for c_idx in range(1, 6):
        cell = ws_resumen.cell(row=row_tot, column=c_idx)
        cell.fill = color_totales
        cell.border = borde_total

    # --- Gráfico de Torta (PieChart) ---
    pie = PieChart()
    pie.title = "Distribución Porcentual del Dinero"
    etiquetas = Reference(ws_resumen, min_col=1, min_row=fila_inicio+1, max_row=row_tot-1)
    datos_dinero = Reference(ws_resumen, min_col=2, min_row=fila_inicio, max_row=row_tot-1)
    pie.add_data(datos_dinero, titles_from_data=True)
    pie.set_categories(etiquetas)
    pie.width = 16
    pie.height = 11
    ws_resumen.add_chart(pie, "G4")

    # Ajuste de anchos de columna
    ws_resumen.column_dimensions['A'].width = 25
    ws_resumen.column_dimensions['B'].width = 22
    ws_resumen.column_dimensions['C'].width = 24
    ws_resumen.column_dimensions['D'].width = 26
    ws_resumen.column_dimensions['E'].width = 20

    # --- Hoja 2: Registros Detallados ---
    ws_detalle = wb.create_sheet(title="Detalle Categorizado")
    ws_detalle.views.sheetView[0].showGridLines = True

    df_export = df[['Fecha de contabilización', 'Número de documento', 'Número de artículo', 'Categoría',
                    'Descripción artículo/serv.', 'Almacén Orígen', 'Almacén destino',
                    'Cantidad', 'Precio Unitario', 'Total', 'Centro de Costos', 'Detalle']].copy()

    headers_det = list(df_export.columns)
    for idx_col, text in enumerate(headers_det, 1):
        cell = ws_detalle.cell(row=1, column=idx_col, value=text)
        cell.fill = color_encabezado
        cell.font = fuente_encabezado
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws_detalle.row_dimensions[1].height = 25

    for r_idx, row in df_export.iterrows():
        n_fila = r_idx + 2
        for c_idx, val in enumerate(row, 1):
            cell = ws_detalle.cell(row=n_fila, column=c_idx, value=val if pd.notna(val) else "")
            nombre_col = headers_det[c_idx-1]

            if nombre_col in ['Total', 'Precio Unitario']:
                cell.number_format = '"$"#,##0.00'
                cell.alignment = Alignment(horizontal="right")
            elif nombre_col in ['Cantidad', 'Número de documento']:
                cell.alignment = Alignment(horizontal="center")
            elif nombre_col == 'Fecha de contabilización':
                cell.alignment = Alignment(horizontal="center")
                if hasattr(val, 'strftime'):
                    cell.value = val.strftime('%Y-%m-%d')
            else:
                cell.alignment = Alignment(horizontal="left")

            cell.border = borde_fino

    for col in ws_detalle.columns:
        max_len = max(len(str(cell.value or '')) for cell in col)
        letra_col = get_column_letter(col[0].column)
        ws_detalle.column_dimensions[letra_col].width = max(max_len + 3, 12)

    # Guardar cambios
    wb.save(path_salida)
    print("✅ ¡Proceso finalizado! El reporte fue generado en la carpeta 'salida/'.")

# Punto de entrada principal
if __name__ == "__main__":
    generar_reporte_almacen()

📖 Paso 1: Leyendo y cargando los datos iniciales...
📊 Paso 2: Calculando sumatorias y porcentajes en dinero...
🎨 Paso 3: Diseñando el reporte en Excel con gráfico y tablas...
✅ ¡Proceso finalizado! El reporte fue generado en la carpeta 'salida/'.
